In [1]:
from pathlib import Path

import cv2
import numpy as np

In [43]:
DATA = "cards6"
SEQUENCE_DIR = Path(f"../data/{DATA}/")
GROUND_TRUTH_PATH = SEQUENCE_DIR / "groundtruth_rect.txt"

OUTPUT_VIDEO_PATH = Path(f"results/csrt_result_{DATA}.mp4")
OUTPUT_PREDICTIONS_PATH = Path(f"results/csrt_predictions_{DATA}.txt")

In [39]:
def load_ground_truth(path: Path) -> np.ndarray:
    """Load tab- or space-separated x, y, width, height boxes."""
    boxes = np.loadtxt(path, delimiter=None, dtype=np.float32)
    boxes = np.atleast_2d(boxes)

    if boxes.shape[1] != 4:
        raise ValueError(
            f"Expected four values per row, but found shape {boxes.shape}"
        )

    return boxes

In [40]:
def create_csrt_tracker():
    """
    Create a CSRT tracker while supporting different OpenCV Python layouts.
    """
    if hasattr(cv2, "TrackerCSRT_create"):
        return cv2.TrackerCSRT_create()

    if hasattr(cv2, "legacy") and hasattr(cv2.legacy, "TrackerCSRT_create"):
        return cv2.legacy.TrackerCSRT_create()

    raise RuntimeError(
        "CSRT is unavailable. Install opencv-contrib-python and make sure "
        "opencv-python is not installed in the same environment."
    )


In [41]:
def calculate_iou(box_a, box_b) -> float:
    """
    Calculate intersection over union.

    Both boxes use:
        x, y, width, height
    """
    ax, ay, aw, ah = map(float, box_a)
    bx, by, bw, bh = map(float, box_b)

    ax2 = ax + aw
    ay2 = ay + ah
    bx2 = bx + bw
    by2 = by + bh

    intersection_left = max(ax, bx)
    intersection_top = max(ay, by)
    intersection_right = min(ax2, bx2)
    intersection_bottom = min(ay2, by2)

    intersection_width = max(0.0, intersection_right - intersection_left)
    intersection_height = max(0.0, intersection_bottom - intersection_top)
    intersection_area = intersection_width * intersection_height

    area_a = max(0.0, aw) * max(0.0, ah)
    area_b = max(0.0, bw) * max(0.0, bh)

    union_area = area_a + area_b - intersection_area

    if union_area <= 0:
        return 0.0

    return intersection_area / union_area

In [31]:
def draw_box(
    frame: np.ndarray,
    box,
    colour,
    label: str,
) -> None:
    x, y, width, height = box

    x = round(x)
    y = round(y)
    width = round(width)
    height = round(height)

    cv2.rectangle(
        frame,
        (x, y),
        (x + width, y + height),
        colour,
        2,
    )

    cv2.putText(
        frame,
        label,
        (x, max(20, y - 8)),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.55,
        colour,
        2,
    )

In [36]:
def main() -> None:
    image_paths = sorted(SEQUENCE_DIR.glob("*.jpg"))
    ground_truth = load_ground_truth(GROUND_TRUTH_PATH)

    if not image_paths:
        raise FileNotFoundError(f"No JPG images found in {SEQUENCE_DIR}")

    if len(image_paths) != len(ground_truth):
        raise ValueError(
            f"Found {len(image_paths)} images but "
            f"{len(ground_truth)} annotation rows."
        )

    first_frame = cv2.imread(str(image_paths[0]))

    if first_frame is None:
        raise RuntimeError(f"Could not read {image_paths[0]}")

    frame_height, frame_width = first_frame.shape[:2]

    # OpenCV 4.13's modern tracker expects built-in integer values:
    # x, y, width, height
    initial_box = tuple(
        int(round(value))
        for value in ground_truth[0]
    )
    
    x, y, width, height = initial_box
    frame_height, frame_width = first_frame.shape[:2]
    
    if width <= 0 or height <= 0:
        raise ValueError(f"Invalid initial box size: {initial_box}")
    
    if x < 0 or y < 0 or x + width > frame_width or y + height > frame_height:
        raise ValueError(
            f"Initial box {initial_box} is outside image dimensions "
            f"{frame_width}x{frame_height}"
        )
    
    print("First frame shape:", first_frame.shape)
    print("Initial box:", initial_box)
    print("Value types:", [type(value) for value in initial_box])
    
    tracker = create_csrt_tracker()
    
    # init() normally returns None on successful initialisation.
    tracker.init(first_frame, initial_box)

    video_writer = cv2.VideoWriter(
        str(OUTPUT_VIDEO_PATH),
        cv2.VideoWriter_fourcc(*"mp4v"),
        20.0,
        (frame_width, frame_height),
    )

    if not video_writer.isOpened():
        raise RuntimeError("Could not create the output video")

    # Frame 1 uses the provided initial box.
    predictions = [initial_box]
    iou_scores = [1.0]

    first_display = first_frame.copy()

    draw_box(
        first_display,
        ground_truth[0],
        (0, 255, 0),
        "Ground truth",
    )
    draw_box(
        first_display,
        initial_box,
        (0, 0, 255),
        "CSRT",
    )

    video_writer.write(first_display)

    for frame_index, image_path in enumerate(image_paths[1:], start=1):
        frame = cv2.imread(str(image_path))

        if frame is None:
            raise RuntimeError(f"Could not read {image_path}")

        tracking_success, predicted_box = tracker.update(frame)

        if tracking_success:
            predicted_box = tuple(map(float, predicted_box))
        else:
            # Simple fallback: retain the previous prediction.
            predicted_box = predictions[-1]

        predictions.append(predicted_box)

        ground_truth_box = ground_truth[frame_index]
        iou = calculate_iou(predicted_box, ground_truth_box)
        iou_scores.append(iou)

        display_frame = frame.copy()

        # Green: ground truth.
        draw_box(
            display_frame,
            ground_truth_box,
            (0, 255, 0),
            "Ground truth",
        )

        # Red: CSRT prediction.
        draw_box(
            display_frame,
            predicted_box,
            (0, 0, 255),
            "CSRT",
        )

        cv2.putText(
            display_frame,
            f"Frame: {frame_index + 1}  IoU: {iou:.3f}",
            (15, 30),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (255, 255, 255),
            2,
        )

        if not tracking_success:
            cv2.putText(
                display_frame,
                "Tracking failure",
                (15, 60),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (0, 0, 255),
                2,
            )

        video_writer.write(display_frame)
        cv2.imshow("CSRT tracking — press Q to quit", display_frame)

        key = cv2.waitKey(1) & 0xFF
        if key == ord("q"):
            print("Display stopped by user. Continuing was cancelled.")
            break

    video_writer.release()
    cv2.destroyAllWindows()

    predictions_array = np.asarray(predictions, dtype=np.float32)

    np.savetxt(
        OUTPUT_PREDICTIONS_PATH,
        predictions_array,
        delimiter="\t",
        fmt="%.0f",
    )

    evaluated_frames = len(iou_scores)
    mean_iou = float(np.mean(iou_scores))

    print(f"Evaluated frames: {evaluated_frames}")
    print(f"Mean IoU: {mean_iou:.4f}")
    print(f"Predictions saved to: {OUTPUT_PREDICTIONS_PATH}")
    print(f"Visualisation saved to: {OUTPUT_VIDEO_PATH}")

In [44]:
if __name__ == "__main__":
    main()

First frame shape: (272, 512, 3)
Initial box: (210, 32, 99, 134)
Value types: [<class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>]
Evaluated frames: 450
Mean IoU: 0.1199
Predictions saved to: csrt_predictions_cards6.txt
Visualisation saved to: csrt_result_cards6.mp4
